# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

fatal: destination path 'SQuAD-explorer' already exists and is not an empty directory.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
import pandas as pd

rows = []

for document in train_data["data"]:
    title = document["title"]

    for index, paragraph in enumerate(document["paragraphs"]):
        rows.append({
            "document_title": title,
            "paragraph_index": index,
            "paragraph_context": paragraph["context"]
        })

In [15]:
parsed = pd.DataFrame(rows)

parsed.head()

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...


In [16]:
parsed.to_csv("parsed.tsv", sep="\t", index=False)

In [17]:
print(parsed.shape)
print(parsed.columns.tolist())
print(parsed.isna().sum())

(18896, 3)
['document_title', 'paragraph_index', 'paragraph_context']
document_title       0
paragraph_index      0
paragraph_context    0
dtype: int64


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    norm="l2",
    stop_words="english"
)

X_tfidf = vectorizer.fit_transform(
    parsed["paragraph_context"]
)

print(X_tfidf.shape)

(18896, 78230)


In [19]:
from sklearn.random_projection import SparseRandomProjection
from sklearn.preprocessing import normalize

projection = SparseRandomProjection(
    n_components=1024,
    random_state=42
)

X_paragraphs = projection.fit_transform(X_tfidf)

X_paragraphs = normalize(X_paragraphs)

print(X_paragraphs.shape)

(18896, 1024)


In [20]:
import json
import numpy as np

# Convert sparse matrix to a dense NumPy array
X_paragraphs = X_paragraphs.toarray()

# Create DataFrame with paragraph identifiers
paragraph_vectors = parsed[
    ["document_title", "paragraph_index"]
].copy()

# Convert each vector into a JSON string
paragraph_vectors["paragraph_vector_json"] = [
    json.dumps(row.tolist())
    for row in X_paragraphs
]

# Save as compressed TSV
paragraph_vectors.to_csv(
    "paragraph-vectors.tsv.gz",
    sep="\t",
    index=False,
    compression="gzip"
)

print(paragraph_vectors.shape)
paragraph_vectors.head()

(18896, 3)


,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.0, 0.0, -0.03406508043403895, 0.0, 0.0, 0.0..."
1,University_of_Notre_Dame,1,"[0.0, 0.0, 0.0, 0.0, 0.03901665175033331, 0.0,..."
2,University_of_Notre_Dame,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,University_of_Notre_Dame,3,"[0.0, 0.0, 0.3154529817666541, 0.0, 0.0, 0.0, ..."
4,University_of_Notre_Dame,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [21]:
import json

paragraph_vectors = parsed[
    ["document_title", "paragraph_index"]
].copy()

paragraph_vectors["paragraph_vector_json"] = [
    json.dumps(row.tolist())
    for row in X_paragraphs
]

paragraph_vectors.to_csv(
    "paragraph-vectors.tsv.gz",
    sep="\t",
    index=False
)

Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [22]:
questions = pd.read_csv("questions.tsv", sep="\t")

questions.head()

,question_id,question
0,1,What was the goal of the abuse of region project?
1,4,How many satellites in the Beidou-1 constellat...
2,7,When did Beyoncé receive ten nominations for ...
3,10,"With which goddess did Sulla, Pompey, and Juli..."
4,13,What area is considered to have a desert clima...


In [23]:
X_questions_tfidf = vectorizer.transform(
    questions["question"]
)

X_questions = projection.transform(X_questions_tfidf)

X_questions = normalize(X_questions)

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [24]:
import json
import numpy as np
import pandas as pd

# Convert sparse question vectors to a dense NumPy array
X_questions = X_questions.toarray()

# Create DataFrame containing question IDs and vectors
question_vectors = pd.DataFrame({
    "question_id": questions["question_id"],
    "question_vector_json": [
        json.dumps(row.tolist())
        for row in X_questions
    ]
})

# Save question vectors to TSV
question_vectors.to_csv(
    "question-vectors.tsv",
    sep="\t",
    index=False
)

# Preview results
question_vectors.head()

,question_id,question_vector_json
0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,13,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [25]:
import faiss
import numpy as np

paragraph_matrix = np.asarray(
    X_paragraphs, dtype="float32"
)

question_matrix = np.asarray(
    X_questions, dtype="float32"
)

index = faiss.IndexFlatL2(1024)

index.add(paragraph_matrix)

distances, indices = index.search(
    question_matrix, 5
)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [26]:
matches = []

for q_idx, question_id in enumerate(questions["question_id"]):
    for rank, paragraph_idx in enumerate(indices[q_idx], start=1):
        paragraph = parsed.iloc[paragraph_idx]

        matches.append({
            "question_id": question_id,
            "question_rank": rank,
            "document_title": paragraph["document_title"],
            "paragraph_index": paragraph["paragraph_index"]
        })

question_matches = pd.DataFrame(matches)

question_matches.to_csv(
    "question-matches.tsv",
    sep="\t",
    index=False
)

Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


In [27]:
# Get the first five questions by question_id
first_five = questions.sort_values("question_id").head(5)

for _, question in first_five.iterrows():

    print("\n" + "=" * 80)
    print("QUESTION ID:", question["question_id"])
    print("QUESTION:", question["question"])
    print("=" * 80)

    matched = question_matches[
        question_matches["question_id"] == question["question_id"]
    ].sort_values("question_rank")

    for _, match in matched.iterrows():

        paragraph = parsed[
            (parsed["document_title"] == match["document_title"]) &
            (parsed["paragraph_index"] == match["paragraph_index"])
        ].iloc[0]

        print("\nRANK:", match["question_rank"])
        print("DOCUMENT:", match["document_title"])
        print("PARAGRAPH INDEX:", match["paragraph_index"])
        print("TEXT:", paragraph["paragraph_context"])
        print("-" * 60)


QUESTION ID: 1
QUESTION: What was the goal of the abuse of region project?

RANK: 1
DOCUMENT: Association_football
PARAGRAPH INDEX: 21
TEXT: In the late 1990s and early 2000s, the IFAB experimented with ways of creating a winner without requiring a penalty shootout, which was often seen as an undesirable way to end a match. These involved rules ending a game in extra time early, either when the first goal in extra time was scored (golden goal), or if one team held a lead at the end of the first period of extra time (silver goal). Golden goal was used at the World Cup in 1998 and 2002. The first World Cup game decided by a golden goal was France's victory over Paraguay in 1998. Germany was the first nation to score a golden goal in a major competition, beating Czech Republic in the final of Euro 1996. Silver goal was used in Euro 2004. Both these experiments have been discontinued by IFAB.
------------------------------------------------------------

RANK: 2
DOCUMENT: Canadian_football

In [28]:
worst_matches = [
    (1, "Canadian_football", 8),
    (4, "BeiDou_Navigation_Satellite_System", 30),
    (7, "Beyoncé", 50),
    (10, "Mary_(mother_of_Jesus)", 6),
    (13, "Sahara", 1)
]

worst_paragraphs = pd.DataFrame(
    worst_matches,
    columns=[
        "question_id",
        "document_title",
        "paragraph_index"
    ]
)

# Validate question IDs
expected_ids = questions.sort_values(
    "question_id"
).head(5)["question_id"].tolist()

assert sorted(worst_paragraphs["question_id"].tolist()) == sorted(expected_ids)

# Verify each selected paragraph was retrieved
for _, row in worst_paragraphs.iterrows():
    valid = question_matches[
        (question_matches["question_id"] == row["question_id"]) &
        (question_matches["document_title"] == row["document_title"]) &
        (question_matches["paragraph_index"] == row["paragraph_index"])
    ]

    assert len(valid) > 0, f"Invalid match for question {row['question_id']}"


Submit "worst-paragraphs.tsv" in Gradescope.

In [29]:
# Export
worst_paragraphs.to_csv(
    "worst-paragraphs.tsv",
    sep="\t",
    index=False
)

print("Validation passed!")
print(worst_paragraphs.to_string(index=False))

Validation passed!
 question_id                     document_title  paragraph_index
           1                  Canadian_football                8
           4 BeiDou_Navigation_Satellite_System               30
           7                            Beyoncé               50
          10             Mary_(mother_of_Jesus)                6
          13                             Sahara                1


Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [30]:
print("Number of rows:", len(worst_paragraphs))
print("Columns:", worst_paragraphs.columns.tolist())
print("Missing values:", worst_paragraphs.isna().sum().sum())

Number of rows: 5
Columns: ['question_id', 'document_title', 'paragraph_index']
Missing values: 0


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

In [31]:
acknowledgements = pd.DataFrame([
    {
        "source": "DX704 Week 10 Examples",
        "reference": "2025-11-03_live_session.ipynb",
    },
    {
            "source": "DX704 Week 10 Examples",
            "reference": "video_testing_document_vectors_for_classification.ipynb",
        },
        {
                    "source": "DX704 Week 10 Examples",
                    "reference": "video_testing_hashed_document_vectors_for_classification.ipynb",
                },
                {
                                    "source": "DX704 Week 10 Examples",
                                    "reference": "video_testing_normalized_document_vectors_for_classification.ipynb",
                                }
])

acknowledgements.to_csv(
    "acknowledgements.txt",
    sep="\t",
    index=False
)